In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:


combined_df = pd.read_csv('clean_data2023.csv')
# combined_df = combined_df.dropna(subset=['P_IV'])
combined_df = combined_df.dropna(subset=['STRIKE'])
combined_df = combined_df.dropna(subset=['UNDERLYING_LAST'])
combined_df = combined_df.dropna(subset=['DTE'])
combined_df = combined_df[combined_df['DTE'] != 0]


k = combined_df['Log-Strike']
T = combined_df['DTE']/365



kappa1 = 5.5
kappa2 = 0.1

rho, eta, gamma, V, V_prime, theta = params
# Function for the SSVI implied volatility

def w(rho, eta, gamma, V, V_prime, theta, k, T):
    theta_T = theta * T + (V - theta) * (1-np.exp(-kappa1 * T)) / kappa1 + (V_prime - theta) * kappa1 * ((1-np.exp(-kappa2 * T)) / kappa2 - (1-np.exp(-kappa1 * T)) / kappa1 ) / (kappa1 - kappa2)
    phi_theta_T = eta / (theta_T ** gamma)
    ssvi_total_variance = (theta_T / 2) * (1 + rho * phi_theta_T * k + np.sqrt((phi_theta_T * k + rho) ** 2 + (1 - rho ** 2)))
    return ssvi_total_variance
    
def d_w_d_T(rho, eta, gamma, V, V_prime, theta, k, T):
    theta_T = theta * T + (V - theta) * (1-np.exp(-kappa1 * T)) / kappa1 + (V_prime - theta) * kappa1 * ((1-np.exp(-kappa2 * T)) / kappa2 - (1-np.exp(-kappa1 * T)) / kappa1 ) / (kappa1 - kappa2)
    phi_theta_T = eta / (theta_T ** gamma)
    d_theta_t_dT = theta + (V - theta)*np.exp(-kappa1*T) + (V_prime - theta) * (kappa1/(kappa1-kappa2)) * (np.exp(-kappa2*T)-np.exp(-kappa1*T))
    first_term = 0.5*(1 + rho*phi_theta_T*k + np.sqrt((phi_theta_T*k + rho)**2 + (1-rho**2)))
    second_term = -rho*eta*gamma*theta_T**(-gamma-1)*k + (-(phi_theta_T * k + rho)*eta*gamma * theta_T**(-gamma-1)*k / np.sqrt((phi_theta_T*k+rho)**2 + (1-rho**2)))
    dwdt = d_theta_t_dT * (0.5 * first_term + 0.5 * theta_T * second_term)
    return dwdt


def d_w_d_k(rho, eta, gamma, V, V_prime, theta, k, T):
    theta_T = theta * T + (V - theta) * (1-np.exp(-kappa1 * T)) / kappa1 + (V_prime - theta) * kappa1 * ((1-np.exp(-kappa2 * T)) / kappa2 - (1-np.exp(-kappa1 * T)) / kappa1 ) / (kappa1 - kappa2)
    phi_theta_T = eta / (theta_T ** gamma)
    dwdk = 0.5 * theta_T * (rho*phi_theta_T+ ((phi_theta_T * k + rho)*phi_theta_T)/(np.sqrt((phi_theta_T*k+rho)**2 + (1-rho**2))))
    return dwdk

def d2w_dk2(rho, eta, gamma, V, V_prime, theta, k, T):
    theta_T = theta * T + (V - theta) * (1-np.exp(-kappa1 * T)) / kappa1 + (V_prime - theta) * kappa1 * ((1-np.exp(-kappa2 * T)) / kappa2 - (1-np.exp(-kappa1 * T)) / kappa1 ) / (kappa1 - kappa2)
    phi_theta_T = eta / (theta_T ** gamma)
    sol = 0.5 * theta_T * ((phi_theta_T**2 * (1-rho**2))/(((phi_theta_T * k + rho)**2 + (1-rho**2))**(3/2)))
    return sol


def local_vol(rho, eta, gamma, V, V_prime, theta, k, T):
    ans = d_w_d_T(rho, eta, gamma, V, V_prime, theta, k, T)/\
    (((k/(2*w(rho, eta, gamma, V, V_prime, theta, k, T)))*\
      (d_w_d_k(rho, eta, gamma, V, V_prime, theta, k, T))-1)**2 \
     + 0.5* d2w_dk2(rho, eta, gamma, V, V_prime, theta, k, T) - \
     0.25*(0.25+1/w(rho, eta, gamma, V, V_prime, theta, k, T))*\
     (d_w_d_k(rho, eta, gamma, V, V_prime, theta, k, T))**2)
    return np.sqrt(ans)

In [ ]:
%matplotlib widget

# SSVI parameters

params = [-0.69153156, 1.32889811, 0.5,2.693e-02, 3.732e-02, 1.100e-01]

rho, eta, gamma, V, V_prime, theta = params

k_range = np.linspace(-2, 2, 200)  # Adjust the range and number of points as needed
T_range = np.linspace(0.1, 3, 200)  # Adjust the range and number of points as needed

K, T = np.meshgrid(k_range, T_range)
LV = np.zeros_like(K)

# Calculate local volatility for each (k, T) pair
for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        LV[i, j] = local_vol(rho, eta, gamma, V, V_prime, theta, K[i, j], T[i, j])

# Plotting the local volatility
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(K, T, LV, cmap='viridis')

ax.set_xlabel('Log-Strike (k)')
ax.set_ylabel('Time to Expiry (T)')
ax.set_zlabel('Local Volatility')

plt.show()




In [ ]:

%matplotlib widget

# SSVI parameters

params = [-0.72669427, 1.1998262, 0.5, 1.355e-02, 3.302e-02, 1.100e-01]


rho, eta, gamma, V, V_prime, theta = params

k_range = np.linspace(-2, 2, 200)  # Adjust the range and number of points as needed
T_range = np.linspace(0.1, 3, 200)  # Adjust the range and number of points as needed

K, T = np.meshgrid(k_range, T_range)
LV = np.zeros_like(K)

# Calculate local volatility for each (k, T) pair
for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        LV[i, j] = local_vol(rho, eta, gamma, V, V_prime, theta, K[i, j], T[i, j])

# Plotting the local volatility
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(K, T, LV, cmap='viridis')

ax.set_xlabel('Log-Strike (k)')
ax.set_ylabel('Time to Expiry (T)')
ax.set_zlabel('Local Volatility')

plt.show()





In [ ]:
%matplotlib widget

# SSVI parameters

params = [-0.65123362, 0.41200321, 0.74083532, 0.03897386, 0.03987186, 0.10999981]  # Paper


rho, eta, gamma, V, V_prime, theta = params

k_range = np.linspace(-2, 2, 200)  # Adjust the range and number of points as needed
T_range = np.linspace(0.1, 3, 200)  # Adjust the range and number of points as needed

K, T = np.meshgrid(k_range, T_range)
LV = np.zeros_like(K)

# Calculate local volatility for each (k, T) pair
for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        LV[i, j] = local_vol(rho, eta, gamma, V, V_prime, theta, K[i, j], T[i, j])

# Plotting the local volatility
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(K, T, LV, cmap='viridis')

ax.set_xlabel('Log-Strike (k)')
ax.set_ylabel('Time to Expiry (T)')
ax.set_zlabel('Local Volatility')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Assuming local_vol is a function that calculates local volatility given parameters and (k, T)
# def local_vol(rho, eta, gamma, V, V_prime, theta, k, T):
#     # Implement the calculation based on your model
#     return calculated_local_volatility

# SSVI parameters for three different sets
params1 = [-0.69153156, 1.32889811, 0.5, 2.693e-02, 3.732e-02, 1.100e-01]
params2 = [-0.72669427, 1.1998262, 0.5, 1.355e-02, 3.302e-02, 1.100e-01]
params3 = [-0.65123362, 0.41200321, 0.74083532, 0.03897386, 0.03987186, 0.10999981]  # Paper

k_range = np.linspace(-2, 2, 200)  # Adjust the range and number of points as needed
T_range = np.linspace(0.1, 3, 200)  # Adjust the range and number of points as needed

K, T = np.meshgrid(k_range, T_range)

# Initialize matplotlib widget if not already done
%matplotlib widget

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot each parameter set
for params in [params1, params2, params3]:
    rho, eta, gamma, V, V_prime, theta = params
    LV = np.zeros_like(K)
    for i in range(K.shape[0]):
        for j in range(K.shape[1]):
            LV[i, j] = local_vol(rho, eta, gamma, V, V_prime, theta, K[i, j], T[i, j])
    ax.plot_surface(K, T, LV, cmap='viridis', alpha=0.5)  # Adjust alpha for better visibility

ax.set_xlabel('Log-Strike (k)')
ax.set_ylabel('Time to Expiry (T)')
ax.set_zlabel('Local Volatility')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming rho, eta, gamma, V, V_prime, theta are defined
# and your local_vol function is correctly implemented
params = [-0.72669427, 1.1998262, 0.5, 1.355e-02, 3.302e-02, 1.100e-01]
rho, eta, gamma, V, V_prime, theta = params

# Your existing code to compute LV
k_range = np.linspace(-2, 2, 100)
T_range = np.linspace(0.1, 3, 100)
K, T = np.meshgrid(k_range, T_range)
LV = np.zeros_like(K)

for i in range(K.shape[0]):
    for j in range(K.shape[1]):
        LV[i, j] = local_vol(rho, eta, gamma, V, V_prime, theta, K[i, j], T[i, j])

# Choose fixed T values for 2D plots, e.g., T=0.5, 1, 2
fixed_T_values = [0.5, 1, 1.5, 2]

# Create a single figure for all plots
plt.figure(figsize=(10, 6))

# Plotting 2D plots for each fixed T in the same figure
for fixed_T in fixed_T_values:
    # Find the index in T_range that is closest to the desired fixed T value
    T_index = np.argmin(np.abs(T_range - fixed_T))
    
    # Extract the local volatility slice for the fixed T
    LV_slice = LV[T_index, :]
    
    # Plotting on the same figure
    plt.plot(k_range, LV_slice, label=f'T={fixed_T}')

# Finalizing the plot
plt.xlabel('Log-Strike (k)')
plt.ylabel('Local Volatility')
plt.title('Local Volatility vs Log-Strike (k) for different T')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as si

# Load your data
combined_df = pd.read_csv('clean_data2023.csv')
# Ensure 'Log-Strike', 'DTE', etc., are correctly calculated

# Function to price European call options using Monte Carlo with local volatility
def monte_carlo_local_vol(S0, K, T, r, N_simulations, N_steps):
    dt = T / N_steps
    S_paths = np.zeros((N_simulations, N_steps + 1))
    S_paths[:, 0] = S0
    
    for t in range(1, N_steps + 1):
        z = np.random.standard_normal(N_simulations)
        # For each step, find local volatility and update the path
        for i in range(N_simulations):
            lv = local_vol(rho, eta, gamma, V, V_prime, theta, np.log(S_paths[i, t-1]/S0), t*dt)
            S_paths[i, t] = S_paths[i, t-1] * np.exp((r - 0.5 * lv**2) * dt + lv * np.sqrt(dt) * z[i])
    
    # Calculate the payoff for each path, then discount back to present value
    payoffs = np.maximum(S_paths[:, -1] - K, 0)
    option_price = np.exp(-r * T) * np.mean(payoffs)
    return option_price

# Black-Scholes price for comparison
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = S * si.norm.cdf(d1, 0.0, 1.0) - K * np.exp(-r * T) * si.norm.cdf(d2, 0.0, 1.0)
    return call_price

# Parameters for the option
S0 = 105  # Spot price
K = 100   # Strike price
T = 1     # Time to maturity (1 year)
r = 0.05  # Risk-free rate
N_simulations = 10000
N_steps = 100

# Local volatility model pricing
mc_price_lv = monte_carlo_local_vol(S0, K, T, r, N_simulations, N_steps)

# Black-Scholes pricing using an estimated constant volatility (for example, using ATM implied volatility)
constant_vol = 0.2  # Placeholder for constant volatility
bs_price = black_scholes_call(S0, K, T, r, constant_vol)

print(f"Monte Carlo Price with Local Volatility: {mc_price_lv}")
print(f"Black-Scholes Price with Constant Volatility: {bs_price}")
